# L12 – RNN och LSTM för sekvenser

**Tid:** ca 2 h · **Del C: Nätverkstyper**

## Mål
1. Förstå idén med **rekurrenta nät**: ett minne som uppdateras steg för steg.
2. Förstå **varför LSTM och GRU** används i stället för en enkel RNN.
3. Göra om en tidsserie till **fönster**, med shape `(batch, tid, features)`.
4. Dela upp tidsserier **efter tid**, inte slumpmässigt, och jämföra mot **naiva baslinjer**.

**Var i arbetsflödet?** Hela flödet, men med två viktiga skillnader för tidsserier: **uppdelningen** (steg 2) och **baslinjen** (steg 3).

> **Körtid:** under 1 minut på GPU, några minuter på CPU.

In [ ]:
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

def set_seed(seed: int = 42) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
CSV = "../../AI25-Deep_learning/Data/CTA_-_Ridership_-_Daily_Boarding_Totals_20260118.csv"

Den här gången använder vi inte `dlkit.fit`. Den är skriven för klassificering (accuracy). För regression skriver vi en kort loop själva. Det är också bra repetition av L03.

## 1. Data: dagliga resor i Chicagos kollektivtrafik

Datan är antalet påstigningar per dag (buss och tåg) hos Chicago Transit Authority. Vi använder **2010–2019**. Pandemin 2020 ändrade resandet i grunden, och en modell tränad före 2020 kan inte förutse det. Strukturella brott av det slaget är ett eget, svårare problem.

In [ ]:
df = pd.read_csv(CSV, thousands=",", parse_dates=["service_date"], date_format="%m/%d/%Y")
df = df.sort_values("service_date").set_index("service_date")
series = df.loc["2010":"2019", "total_rides"].asfreq("D") / 1e6    # miljoner resor per dag
print(f"{len(series)} dagar, saknade värden: {series.isna().sum()}")

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
series.plot(ax=axes[0], title="Totalt antal resor per dag (miljoner)")
series["2018-03":"2018-05"].plot(ax=axes[1], marker=".", title="Tre månader: tydligt veckomönster")
plt.tight_layout(); plt.show()

Vi ser:
- ett **veckomönster**: vardagar cirka 1,6 miljoner, helger betydligt färre,
- **helgdagar** som avviker,
- en **långsam trend** nedåt under åren.

## 2. Uppdelning efter tid

**Varför inte `train_test_split` med shuffle?** Då hamnar dagar från 2019 i träningsdatan och dagar från 2015 i testdatan. Modellen får "titta in i framtiden", och resultatet kan bli för optimistiskt (övning 3 undersöker hur mycket). Det är dataläckage (L01). I verkligheten förutsäger man alltid framtiden utifrån det förflutna, och uppdelningen ska spegla det.

**sklearn:** `TimeSeriesSplit` gör samma sak för korsvalidering. Valideringsdelen ligger alltid *efter* träningsdelen i tid.

In [ ]:
TRAIN_END, VAL_END = "2016-12-31", "2018-12-31"   # train 2010–2016, val 2017–2018, test 2019

# Standardisera med statistik från ENBART träningsperioden
mean, std = series[:TRAIN_END].mean(), series[:TRAIN_END].std()
z = ((series - mean) / std).to_numpy(dtype="float32")
print(f"mean={mean:.3f} M, std={std:.3f} M")

## 3. Fönster: från tidsserie till (X, y)

En modell behöver exempel i formen (indata, mål). För en tidsserie skapar vi dem med ett **glidande fönster**: de senaste `W` dagarna är indata, och nästa dag är målet.
```
dag:   1  2  3  4  5  6  7  8  9 ...
       [-- X (W=7) --]  y
          [-- X (W=7) --]  y
```
Formatet för RNN i PyTorch med `batch_first=True` är **`(batch, tid, features)`**. Här är det `(N, W, 1)`: en feature (antal resor) per tidssteg.

In [ ]:
WINDOW = 28   # fyra veckor bakåt, så att veckomönstret syns flera gånger

def make_windows(values, window):
    X = np.stack([values[i - window:i] for i in range(window, len(values))])
    y = values[window:]
    return X[..., None], y          # X: (N, window, 1), y: (N,)

X, y = make_windows(z, WINDOW)
target_dates = series.index[WINDOW:]
is_train = target_dates <= TRAIN_END
is_val = (target_dates > TRAIN_END) & (target_dates <= VAL_END)
is_test = target_dates > VAL_END
print("X:", X.shape, " train/val/test:", is_train.sum(), is_val.sum(), is_test.sum())

Valideringsfönster i början av 2017 innehåller dagar från slutet av 2016. Det är tillåtet, eftersom det bara är *förfluten* information. Det viktiga är att *målen* ligger i rätt period.

## 4. Baslinjer

Innan vi tränar något behöver vi veta vad som är bra (L01). För tidsserier finns två naturliga baslinjer:
- **Naiv:** imorgon blir som idag.
- **Säsongsnaiv:** imorgon blir som samma veckodag förra veckan.

In [ ]:
def to_rides(z_values):
    return z_values * std + mean    # tillbaka till miljoner resor

def mae_rides(pred_z, true_z):
    return np.abs(to_rides(pred_z) - to_rides(true_z)).mean() * 1e6   # i antal resor

y_val = y[is_val]
mae_naive = mae_rides(X[is_val, -1, 0], y_val)          # sista dagen i fönstret
mae_seasonal = mae_rides(X[is_val, -7, 0], y_val)       # samma veckodag förra veckan
print(f"Naiv (igår):                 MAE {mae_naive:>9,.0f} resor/dag")
print(f"Säsongsnaiv (förra veckan):  MAE {mae_seasonal:>9,.0f} resor/dag")

Den säsongsnaiva baslinjen är mycket bättre, eftersom veckomönstret dominerar. **Det är den vi måste slå.** Om ett neuralt nät inte slår den är det inte värt besväret.

## 5. Rekurrenta nät

### Idén
En RNN läser sekvensen **ett steg i taget** och uppdaterar ett **dolt tillstånd** $h_t$, ett slags minne:

$$h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$$

- **Samma vikter** används i varje tidssteg, som de delade vikterna i en CNN men längs tiden. Därför fungerar den för sekvenser av godtycklig längd.
- Efter sista steget sammanfattar $h_T$ hela sekvensen. Ett `Linear`-lager gör prediktionen utifrån det.

### Varför LSTM?
Under backward multipliceras gradienten genom **alla tidssteg**. Med 28 steg krymper den lätt till nästan noll (försvinnande gradienter, L05), och en enkel RNN glömmer då det som hände långt bak.

**LSTM** (Long Short-Term Memory) har ett separat **cellminne** och tre **grindar**, små sigmoid-lager med värden mellan 0 och 1, som styr flödet:
- **forget gate:** vad i minnet ska glömmas?
- **input gate:** vad nytt ska sparas?
- **output gate:** vad av minnet ska användas just nu?

Eftersom minnet uppdateras genom *addition* i stället för upprepad multiplikation överlever gradienten många fler tidssteg. **GRU** är en förenklad variant med två grindar. Den är ofta lika bra och lite snabbare.

In [ ]:
class Forecaster(nn.Module):
    def __init__(self, cell=nn.LSTM, input_size=1, hidden_size=64):
        super().__init__()
        self.rnn = cell(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):                 # x: (N, W, features)
        out, _ = self.rnn(x)              # out: (N, W, hidden), det dolda tillståndet i varje steg
        return self.head(out[:, -1])      # använd bara det SISTA steget: (N, 1)

m = Forecaster()
print(m)
print("utdata:", m(torch.randn(8, WINDOW, 1)).shape)

## 6. Träning

En träningsloop för regression med `MSELoss`. Early stopping görs på det enklaste sättet: vi behåller vikterna från epoken med lägst val loss (L07).

In [ ]:
def train_forecaster(model, X_tr, y_tr, X_va, y_va, epochs=40, lr=1e-3, batch_size=64, verbose=True):
    model = model.to(device)
    Xt, yt = torch.tensor(X_tr, device=device), torch.tensor(y_tr, device=device)[:, None]
    Xv, yv = torch.tensor(X_va, device=device), torch.tensor(y_va, device=device)[:, None]
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    best_loss, best_state, history = float("inf"), None, {"train_loss": [], "val_loss": []}
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(Xt), device=device)   # blanda FÖNSTREN. Ordningen inuti varje fönster bevaras.
        total = 0.0
        for i in range(0, len(Xt), batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(Xt[idx]), yt[idx])
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xv), yv).item()
        history["train_loss"].append(total / len(Xt)); history["val_loss"].append(val_loss)
        if val_loss < best_loss:
            best_loss, best_state = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    if verbose:
        print(f"bästa val loss {best_loss:.4f} vid epok {int(np.argmin(history['val_loss'])) + 1}")
    return history

@torch.no_grad()
def forecast(model, X_):
    model.eval()
    return model(torch.tensor(X_, device=device)).cpu().numpy()[:, 0]

**Varför får vi blanda nu, när vi inte fick blanda vid uppdelningen?** Inuti varje fönster bevaras tidsordningen, och det är den modellen lär sig av. Att träningsexemplen (fönstren) visas i slumpvis ordning påverkar inte vad som är "framtid". Alla mål ligger fortfarande i träningsperioden.

In [ ]:
set_seed(42)
lstm = Forecaster(nn.LSTM)
h = train_forecaster(lstm, X[is_train], y[is_train], X[is_val], y[is_val])
mae_lstm = mae_rides(forecast(lstm, X[is_val]), y_val)

plt.figure(figsize=(8, 3.5))
plt.plot(h["train_loss"], label="train"); plt.plot(h["val_loss"], label="val")
plt.yscale("log"); plt.xlabel("epok"); plt.ylabel("MSE (standardiserad)"); plt.legend(); plt.show()
print(f"LSTM:        MAE {mae_lstm:>9,.0f}")
print(f"Säsongsnaiv: MAE {mae_seasonal:>9,.0f}")

LSTM:en slår den säsongsnaiva baslinjen med cirka 25 %. Vi tittar på prediktionerna:

In [ ]:
pred_val = pd.Series(to_rides(forecast(lstm, X[is_val])), index=target_dates[is_val])
true_val = pd.Series(to_rides(y_val), index=target_dates[is_val])

fig, ax = plt.subplots(figsize=(12, 4))
true_val["2017-11":"2018-01"].plot(ax=ax, label="verklighet", marker=".")
pred_val["2017-11":"2018-01"].plot(ax=ax, label="LSTM", marker=".")
ax.set_ylabel("miljoner resor"); ax.legend(); ax.set_title("Val: november 2017 – januari 2018")
plt.show()

errors = (pred_val - true_val).abs().sort_values(ascending=False)
print("Största felen (felanalys, L06):")
print((errors.head(8) * 1e6).round(-3).astype(int).to_string())

**Felanalys:** de största felen ligger på **helgdagar** som jul, Thanksgiving, Labor Day, Memorial Day och nyår. Modellen ser bara antalet resor de senaste 28 dagarna och har ingen information om att det är en helgdag. Det pekar på nästa förbättring: **lägg till features** som veckodag eller helgdag (övning 4). Den sortens insikt får man från felanalys, inte från ett mått.

## 7. Vad händer om … vi byter cell?

**Gissa först:** hur står sig en enkel `nn.RNN` och en `nn.GRU` mot LSTM:en på 28 tidssteg?

In [ ]:
for cell in [nn.RNN, nn.GRU, nn.LSTM]:
    set_seed(42)
    model = Forecaster(cell)
    train_forecaster(model, X[is_train], y[is_train], X[is_val], y[is_val], verbose=False)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{cell.__name__:5s} {n_params:>7,} parametrar   val MAE {mae_rides(forecast(model, X[is_val]), y_val):>9,.0f}")

Med 28 steg och ett tydligt veckomönster klarar sig alla tre ungefär lika bra. Skillnaden mellan RNN och LSTM syns först vid **långa beroenden**, hundratals steg. GRU och LSTM är ändå standardvalen, eftersom de är robustare. LSTM har cirka 4 gånger fler parametrar än en RNN (fyra uppsättningar vikter, en per grind plus kandidaten) och GRU cirka 3 gånger fler.

## 8. Test – en gång

In [ ]:
mae_test_lstm = mae_rides(forecast(lstm, X[is_test]), y[is_test])
mae_test_seasonal = mae_rides(X[is_test, -7, 0], y[is_test])
print(f"TEST 2019 – LSTM: {mae_test_lstm:,.0f}   säsongsnaiv: {mae_test_seasonal:,.0f}")

## 9. Övningar

### Övning 1 – fönster
Kontrollera att du förstår `make_windows`. Vad blir `X` och `y` för serien `[0, 1, 2, 3, 4, 5]` med `window=3`? Skriv svaret som arrayer.

In [ ]:
expected_X = None   # TODO: shape (3, 3, 1)
expected_y = None   # TODO: shape (3,)

In [ ]:
Xw, yw = make_windows(np.arange(6, dtype="float32"), 3)
assert np.array_equal(Xw, np.array(expected_X, dtype="float32")) and np.array_equal(yw, np.array(expected_y, dtype="float32"))
print("Rätt!")

### Övning 2 – fönsterlängd
Prova `WINDOW = 7` och `WINDOW = 56`. Hur påverkas val MAE? Varför är 7 en riskabel längd för just den här datan?

*Ledtråd:* du behöver göra om `make_windows` och masken per period med den nya längden.

In [ ]:
# TODO

*Din tolkning:*

### Övning 3 – slumpvis uppdelning: vad mäter den egentligen?
Dela fönstren 2010–2018 slumpvis i 80 % train och 20 % val (som `train_test_split`) i stället för efter tid, och träna en LSTM. Skriv ut **både val MAE och test-MAE (2019)** för den modellen och för den tidsbaserade.
1. Hur väl förutsäger val-siffran test-siffran i de två fallen?
2. Är den slumpvisa uppdelningen ett problem för just den här modellen och datan? När blir den det?

In [ ]:
# TODO

*Din förklaring:*

### Övning 4 – lägg till veckodag som feature
Ge modellen **veckodagen för varje dag i fönstret** som 7 extra features (one-hot), så att `input_size=8`. Slå LSTM:en utan veckodag på val MAE.

*Ledtråd:* `series.index.day_of_week` och `np.eye(7)[...]` ger one-hot. Bygg fönster av en matris med shape `(dagar, 8)`.

In [ ]:
# TODO
mae_with_dow = None

In [ ]:
assert mae_with_dow < mae_lstm, (mae_with_dow, mae_lstm)
print(f"Rätt! {mae_with_dow:,.0f} jämfört med {mae_lstm:,.0f}")

## När använder jag detta?
- **Tidsbaserad uppdelning:** *alltid* för tidsserier och för data där framtiden ska förutsägas.
- **Naiva baslinjer** (igår, förra veckan): alltid först. De är förvånansvärt svåra att slå.
- **LSTM och GRU:** för sekvenser av måttlig längd (tiotals till några hundra steg): tidsserier, sensordata, enklare textuppgifter. GRU är ett bra standardval och LSTM ett robust alternativ.
- **Transformers** (L13): för långa sekvenser och text, när det finns mycket data.
- **Tabellmodeller** (till exempel gradient boosting med lag-features som "förra veckan"): ofta lika bra eller bättre för tidsserier, och värda att jämföra med.

## Sammanfattning
- En RNN läser ett steg i taget och uppdaterar ett dolt tillstånd. LSTM och GRU har grindar som gör att gradienterna överlever fler steg.
- Indata har shapen `(batch, tid, features)` med `batch_first=True`.
- Tidsserier delas upp efter tid. Fönster får blandas inom train.
- Felanalysen visade att helgdagar är det svåra, och det pekar på bättre features.

**Nästa lektion:** attention och transformers, arkitekturen bakom dagens språkmodeller.